# Variables, Naming Conventions, and Dynamic Typing

**Topic:** `1.4` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*A name is a label, not a box. Binding, aliasing, shared mutation, and the naming conventions that keep a fleet's code readable to the next engineer.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Almost every confusing bug in a Python codebase comes from one of three facts on this page:

1. **Assignment does not copy** — it only attaches a name to an object.
2. **Names can be rebound** to a value of a completely different type.
3. **Two names can share one object**, so mutation through one is visible through the other.

Robotics code passes values between stages constantly. These three facts decide whether a pipeline is predictable or full of subtle state bugs.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain the difference between a **name** and the **object** it refers to.
2. Predict the effect of rebinding a name to a value of another type.
3. Demonstrate **aliasing** and choose correctly between copy and reference.
4. Apply **PEP 8 naming** for functions, classes, constants, and internals.
5. Avoid shadowing built-ins and identify the failure mode when it happens.
6. Validate and cast configuration values once at a system boundary.

## 3. Prerequisites

This topic assumes you already have:

- Topic 1.1 — Introduction to Python
- Topic 1.3 — Syntax, Indentation and Comments

## 4. Mental Model: Sticky Notes on Objects

**The object is the thing with the data. A name is a sticky note you attach to it.**

You can move a sticky note to another object at any time, and you can attach several notes to the same object. Nothing is ever put *into* the name.

| Operation | What a beginner imagines | What actually happens |
| --- | --- | --- |
| `b = 48.0` | A box of type float is created for `b` | `b` now labels one float object |
| `b = 'x'` | The box changes type | `b` is re-pointed at a str object |
| `a = b` | A second box is filled | `a` and `b` label the **same** object |
| `a.append(...)` | Only `a` changes | the shared object changes |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| **Variable** | A name bound to an object; not a typed container. |
| **Object** | The thing that holds data and carries a type. |
| **Binding** | Attaching a name to an object, or re-pointing it elsewhere. |
| **Namespace** | The mapping from names to objects (globals, locals, …). |
| **Rebinding** | Pointing an existing name at a different object. |
| **Alias** | A second name referring to the same object as a first. |
| **Mutability** | Whether an object's contents can be changed in place. |
| **PEP 8 naming** | snake_case for functions, CapWords for classes, UPPER_CASE for constants. |
| **`is` operator** | Identity comparison: do two names refer to one object? |
| **`id()`** | The memory address of an object (not a durable identity). |

## 6. Conceptual Explanation

In Python a variable is **not** a container that holds a value. It is a **name** bound to an **object**. The object carries its type and its data; the name is a label the interpreter can re-point at any time.

Assignment is therefore a **binding** operation, not a copy. This one distinction explains:

- why two names can refer to the same object,
- why rebinding a name to a different type is perfectly legal, and
- why mutating an object through one name is visible through another.

In [ ]:
battery = 48.0
print(type(battery).__name__, battery)

# rebinding the same name to a different type is legal
battery = 'unknown'
print(type(battery).__name__, battery)

# and back again
battery = 12.0
print(type(battery).__name__, battery)

Two names can point at one object. When that happens, mutating the object through either name is visible through the other — the classic source of bugs where a function unexpectedly modifies its argument.

In [ ]:
original = [1.0, 2.0, 3.0]
alias = original          # same object, not a copy
alias.append(4.0)

print('original:', original)          # also has 4.0
print('same object?', original is alias)

copy = original.copy()    # independent top-level list
copy.append(5.0)
print('after copy:', original)        # unchanged

> **`is` versus `==`**
>
> - `is` asks whether two names point at the *same object*.
> - `==` asks whether two objects *compare equal*.
>
> Small integers and short strings are interned, so `a is b` can be true for two separately written literals. In your own code, prefer `==` for values and reserve `is` for identity checks against `None` or sentinel objects.

## 7. Formal Theory

A Python program maintains two structures at runtime:

1. An **object heap** — where each object stores its type and data.
2. A **namespace** — a mapping from names to objects.

```text
namespace[name]  →  object   (the object owns type + data)
```

Assignment writes one entry into a namespace. `del` removes it. The object's lifetime is governed by how many names (or container slots) refer to it, not by the name itself.

1. The expression on the right of `=` is evaluated first → produces a reference to an object.
2. The name on the left is created or rebound to that object in its namespace.
3. The object is freed only when no namespace entry and no container slot refers to it.

In [ ]:
def show(label, value):
    print(f'{label:<12} id={id(value):<16} type={type(value).__name__}')


a = 48.0
b = a
c = 48.0
show('a', a)
show('b', b)
show('c', c)
print('a is b:', a is b, '| a is c:', a is c, '| a == c:', a == c)

| Operation | Effect on the namespace |
| --- | --- |
| `x = value` | Create or rebind the name `x` |
| `x += value` | Read `x`, mutate or rebind, write `x` back |
| `del x` | Remove the name; the object may survive elsewhere |
| `x: int = 5` | Bind, plus record a type hint in `__annotations__` |

## 8. Syntax & Naming Conventions (PEP 8)

PEP 8 defines four naming styles. Choosing the right one is most of the job of writing readable Python.

In [ ]:
# snake_case for functions and variables
max_speed_mps = 1.5


def clamp_pct(value):
    return max(0, min(100, value))


# UPPER_CASE for module-level constants
LOW_BATTERY_PCT = 20


# CapWords for classes; _leading_underscore marks internals
class TelemetryChannel:
    _counter = 0

    def __init__(self, name):
        self.name = name

The anti-patterns to avoid:

```text
MaxSpeed = 1.5        # CapWords for a constant  → use MAX_SPEED
def getValue(x):      # camelCase for a function → use get_value
X = 2                 # one-letter module constant
```

> **Never shadow a built-in**
>
> `list = [1, 2]` silently rebinds the built-in for the whole module scope. The error then appears far away as `TypeError: 'list' object is not callable`.

## 9. Basic Examples

**Example 1 — Several names bound in one statement**

In [ ]:
robot_id, battery_pct, status = 'rover-01', 48.0, 'ACTIVE'
print(robot_id, battery_pct, status)

# swapping without a temporary variable
x, y = 1, 2
x, y = y, x
print('after swap:', x, y)

**Example 2 — Augmented assignment**

In [ ]:
distance_m = 0.0
distance_m += 1.5
distance_m *= 2
print('distance:', distance_m)

tags = ['front']
tags += ['rear']      # extends in place; same object
print('tags:', tags)

## 10. Intermediate Examples — Unpacking

Unpacking turns a tuple of values into a set of named variables in one step. It reads especially well when the source is already structured (a coordinate pair, a status snapshot, a tuple returned by a function).

In [ ]:
position = (12.0, 3.5)
x, y = position
print(f'x={x} y={y}')

state = {'name': 'rover-01', 'battery_pct': 48.0, 'status': 'ACTIVE'}
name, pct, status = state['name'], state['battery_pct'], state['status']
print(f'{name} {pct}% {status}')

# swapping is the clearest use of simultaneous assignment
low, high = 10, 20
low, high = high, low
print('low:', low, 'high:', high)

## 11. Advanced Examples — Inspecting a Namespace

Because names live in dictionaries, a module's public surface can be inspected rather than remembered. This is how documentation tools and test frameworks discover what a module offers.

In [ ]:
MAX_SPEED_MPS = 1.5


def clamp_pct(value):
    return max(0, min(100, value))


public = sorted(
    name for name in globals()
    if not name.startswith('_') and name.isidentifier()
)
print('public names:', public)

Note that the private convention is *by agreement*, not enforcement: `_name` is still fully accessible. It signals intent to other engineers — a real but weaker guarantee than a compiler check.

## 12. Code Walkthrough — A Configuration Loader

Below is a small configuration loader. It is the first place where naming and binding discipline matter together: the loader converts values **once, at the boundary**, so that everything downstream can trust what it receives.

In [ ]:
"""config_loader.py – turn raw settings into trusted values."""
from __future__ import annotations

DEFAULTS = {
    'robot_id': 'rover-01',
    'max_speed_mps': 1.5,
    'battery_wh': 48.0,
}

CASTS = {
    'robot_id': str,
    'max_speed_mps': float,
    'battery_wh': float,
}


class ConfigError(ValueError):
    """Raised when a setting cannot be converted or is out of range."""


def load_config(raw: dict) -> dict:
    """Merge raw settings over the defaults and validate each value."""
    config = dict(DEFAULTS)          # copy so module defaults stay clean
    config.update(raw)
    for key in list(config):
        cast = CASTS.get(key)
        if cast is None:
            raise ConfigError(f'unknown setting: {key}')
        try:
            config[key] = cast(config[key])
        except (TypeError, ValueError) as exc:
            raise ConfigError(f'{key}: cannot convert {config[key]!r}') from exc
    if not 0 < config['max_speed_mps'] <= 3.0:
        raise ConfigError('max_speed_mps must be in (0, 3.0]')
    return config

Two details matter:

1. `config = dict(DEFAULTS)` **copies**, so the module-level default is never mutated by a caller.
2. The explicit `from exc` chain preserves the original conversion error, which makes a bad setting diagnosable instead of merely reported.

## 13. Line-by-Line Explanation

1. `DEFAULTS` holds the shipped settings; `dict(DEFAULTS)` inside the function copies them so a caller cannot corrupt the module state.
2. `CASTS` records the target type per setting, making the conversion table explicit.
3. `ConfigError` subclasses `ValueError` so callers can catch either the specific error or the standard one.
4. `config.update(raw)` applies overrides before validation, so validation always sees the final effective value.
5. Iterating over `list(config)` allows the loop body to reassign keys safely.
6. `raise … from exc` keeps the underlying conversion error in the traceback.
7. The final range check on `max_speed_mps` enforces a policy that no type conversion can express.

## 14. Common Mistakes

**Mistake 1 — Assuming assignment copies the object**

```python
a = [1.0, 2.0]
b = a            # same object, not a copy
b.append(3.0)
print(a)         # [1.0, 2.0, 3.0]

b = a.copy()     # explicit copy when independence is intended
```

**Mistake 2 — Mutating a caller's argument in place**

```python
def extend_log(log, entry):
    log.append(entry)      # caller sees the change

def extend_log(log, entry):
    return log + [entry]   # caller is unaffected
```

**Mistake 3 — Shadowing a built-in**

```python
list = [1, 2, 3]
more = list((4, 5))   # TypeError: 'list' object is not callable
# the shadowing lasts for the whole enclosing scope
```

**Mistake 4 — Reusing a single-letter name across an entire module**

```python
def area(w, h):
    return w * h
# at the call site: area(w, h) tells the reader nothing
```

## 15. Debugging Techniques

When a value is not what you expected, three prints settle almost every case: its **type**, its **representation**, and its **identity**.

In [ ]:
config = {'max_speed_mps': '1.5'}
raw = config['max_speed_mps']

print('type :', type(raw).__name__)
print('repr :', repr(raw))
print('id   :', id(raw))

# a string that looks like a number is still a string
try:
    print(raw * 2)
except TypeError as exc:
    print('TypeError:', exc)

To find out who rebound a name, print the namespace rather than guessing: `globals()`, `locals()` and `vars(obj)` are ordinary dictionaries.

In [ ]:
def inspect_scope() -> dict:
    local_var = 1
    return {k: v for k, v in locals().items() if not k.startswith('_')}


print(inspect_scope())

> **`id()` is not an identity you can rely on**
>
> CPython reuses memory addresses after an object is freed, so two objects can share an id at different times. Use `is` for the question you care about, not `id(a) == id(b)`.

## 16. Best Practices

- **snake_case** for functions and variables; **CapWords** for classes.
- **UPPER_CASE** for module-level constants that must not be rebound.
- A single leading underscore marks a name as internal **by convention**.
- Names should read as prose in context: `battery_pct`, not `b`.
- Never shadow a built-in; check the list before naming anything.
- Copy before mutating when the caller may still hold a reference.
- Convert types **once at the boundary**, not at every use site.
- Return a value rather than relying on a side effect the caller cannot see.

## 17. Performance Considerations

Binding a name is cheap: it is a dictionary write. What is *not* cheap is repeated attribute lookup inside a hot loop (`obj.attr` is another dictionary lookup after the name is resolved). Hoisting a value out of a loop is a legitimate micro-optimisation — but only when measurement shows the loop is hot enough to care.

The optimisation that reliably matters in this topic is **avoiding accidental copies**. Rebinding a name to an immutable object costs nothing, but `config = dict(DEFAULTS)` inside a function called ten thousand times creates ten thousand dictionaries. When a default must not be mutated across calls, make the copy once at module scope and treat it as immutable by convention.

> **Measure the loop, not the line**
>
> Use `time.perf_counter` around a realistic workload. A micro-optimisation that shaves 2 % from a loop executed once per mission is rarely worth the readability cost.

## 18. Pythonic Approaches

In [ ]:
# swapping: temporary variable versus simultaneous assignment
a, b = 'front', 'rear'
a, b = b, a

# copying a list: slice versus copy versus list()
readings = [0.1, 0.2]
shallow = readings[:]
explicit = readings.copy()
via_ctor = list(readings)
print(shallow, explicit, via_ctor)

# building a string from pieces
parts = ['rover-01', 'ready']
print('-'.join(parts))
print(''.join(['batt', 'ery']))

## 19. Robotics Connection

Robot state flows between names constantly: a sensor read is bound to a name, passed to a function, and compared against a threshold. Because binding does not copy, a function that mutates a list argument changes the caller's data — which is why the telemetry pipeline in this course always copies at the boundary.

In [ ]:
# Lightweight stand-in for the course simulator (no external hardware needed)
class SimulatedRobot:
    def __init__(self, name='rover-01', battery_wh=48.0):
        self.name = name
        self.battery_wh = battery_wh

    def status(self):
        return {
            'name': self.name,
            'battery_wh': self.battery_wh,
            'voltage_v': round(self.battery_wh / 4.0, 2),
        }


robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
snapshot = robot.status()

# two names, one object – mutating the snapshot changes what we hold
alias = snapshot
print('same object?', alias is snapshot)
snapshot['note'] = 'captured at boot'
print('visible via alias:', alias.get('note'))

## 20. Engineering Example — Bound Names via Bytecode

A small helper that records every name a function binds — the practical form of “what changed in here?”

In [ ]:
def bound_names(func) -> set[str]:
    """Return the local names a function binds, via its code object."""
    import dis
    names = set()
    for instruction in dis.get_instructions(func):
        if instruction.opname in {'STORE_NAME', 'STORE_FAST'}:
            names.add(instruction.argval)
    return names


def sample():
    battery_pct = 48.0
    status = 'READY'
    return battery_pct, status


print(sorted(bound_names(sample)))

## 21. Guided Practice

Try these before moving to the exercises.

1. Bind `robot_id` to a string and `battery_pct` to a float; print type, repr and id for both.
2. Create two names for the same list, append through one, and print both to show the change is shared.
3. Create a genuine copy and repeat the append to show independence.
4. Rebind `battery_pct` to the string `'unknown'` and explain why nothing complains until it is used in arithmetic.

In [ ]:
# Your turn – fill in the blanks or run the starter code below.
robot_id = 'rover-01'
battery_pct = 48.0
print(type(robot_id).__name__, repr(robot_id))
print(type(battery_pct).__name__, repr(battery_pct))

shared = [0.1]
also_shared = shared
also_shared.append(0.2)
print('shared:', shared)

independent = shared.copy()
independent.append(0.3)
print('shared after copy:', shared)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

1. After `b = a` where `a` is a list, what is true?
2. Why convert values to their final types once, at the boundary of a program?
3. Which change makes a function safe to call with a list the caller wants to keep?
4. Why does the telemetry pipeline copy incoming readings at the boundary?

## 23. Summary

A Python variable is a **name bound to an object**. Assignment binds or rebinds; it never copies. Because of that:

- two names can refer to one object,
- mutating through one name is visible through the other, and
- rebinding a name to a value of a different type is completely legal.

Dynamic typing is the direct consequence: the type lives on the object and is checked when an operation is attempted, not when the name is created.

The discipline that keeps this safe is boring and effective:

- Name things so a reader understands them from context alone.
- Never shadow a built-in.
- Copy before mutating when the caller may still hold a reference.
- Convert types once at the boundary where data enters your program, then trust them downstream.
- When a value is not what you expected, print its type, its representation and its identity before changing any code.

## 24. Key Takeaways

- A name is a label bound to an object, never a typed box.
- Assignment binds or rebinds; it does not copy.
- Two names can share one object, so mutation is shared.
- Types belong to objects and are checked when an operation runs.
- snake_case for functions, CapWords for classes, UPPER_CASE for constants.
- Never shadow built-ins; the failure surfaces far from its cause.
- Copy explicitly when the caller must keep its own data.
- Diagnose with `type()`, `repr()` and `is` before editing code.

## 25. Further Exploration

- [PEP 8 – Style Guide](https://peps.python.org/pep-0008/) — the naming conventions used throughout this course.
- [Naming and binding in Python](https://realpython.com/python-names-and-scoping/)
- [dataclasses – immutable-ish records](https://docs.python.org/3/library/dataclasses.html)
- [copy – shallow and deep copying](https://docs.python.org/3/library/copy.html)
- [dis – inspecting bytecode and bound names](https://docs.python.org/3/library/dis.html)